# 集成强化学习投资策略 (PPO + A2C + DQN)

**论文参考**: 李方涛 (2022), 《基于强化学习的集成投资策略研究》

**核心思想**: 同时训练PPO、A2C、DQN三个智能体，通过多数投票(majority vote)集成它们的交易决策，累计收益约70%。

**数据**: 沪深300ETF (510300) 日线数据

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. DQN (Deep Q-Network)
基于价值函数的方法，学习动作-价值函数 $Q(s, a)$:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

### 2. A2C (Advantage Actor-Critic)
同时学习策略$\pi_\theta(a|s)$和价值函数$V_\phi(s)$，使用优势函数:

$$A(s, a) = r + \gamma V(s') - V(s)$$

$$\nabla_\theta J = \mathbb{E}\left[ A(s, a) \nabla_\theta \log \pi_\theta(a|s) \right]$$

### 3. PPO (Proximal Policy Optimization)
限制策略更新幅度的Actor-Critic方法:

$$L^{CLIP}(\theta) = \mathbb{E}\left[ \min\left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

其中 $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ 为概率比。

### 4. 集成策略
$$a_{\text{ensemble}} = \text{majority\_vote}(a_{\text{PPO}}, a_{\text{A2C}}, a_{\text{DQN}})$$

In [ ]:
# ============ 动画: 训练奖励曲线逐步改善 ============
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n_episodes = 80
episodes = np.arange(1, n_episodes + 1)

# 模拟三个智能体的训练曲线
def sim_reward_curve(base, noise, trend):
    return base + trend * np.log1p(episodes) + np.cumsum(np.random.randn(n_episodes) * noise) * 0.1

rewards_ppo = sim_reward_curve(-50, 8, 25)
rewards_a2c = sim_reward_curve(-60, 10, 22)
rewards_dqn = sim_reward_curve(-55, 12, 20)

frames = []
for i in range(5, n_episodes + 1, 2):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=episodes[:i], y=rewards_ppo[:i], mode='lines',
                       name='PPO', line=dict(color='#2196F3', width=2)),
            go.Scatter(x=episodes[:i], y=rewards_a2c[:i], mode='lines',
                       name='A2C', line=dict(color='#4CAF50', width=2)),
            go.Scatter(x=episodes[:i], y=rewards_dqn[:i], mode='lines',
                       name='DQN', line=dict(color='#F44336', width=2)),
        ],
        name=f'Episode {i}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='集成RL智能体训练奖励曲线'),
        xaxis_title='Episode', yaxis_title='累计奖励',
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 100}, 'transition': {'duration': 50}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============ 导入与配置 ============
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

from shared.data_fetcher import get_etf_daily
from shared.backtest_engine import Backtester
from shared.visualization import plot_equity_curve, plot_drawdown, plot_metrics_table
from shared.factors import rsi, macd, bollinger_bands, atr
from shared.constants import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============ 数据获取与特征工程 ============
df = get_etf_daily('510300', start_date='20180101', end_date='20241231')
print(f'数据量: {len(df)} 条')

# 技术指标作为状态特征
df['returns'] = df['close'].pct_change()
df['rsi'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb = bollinger_bands(df['close'])
df['bb_pctb'] = bb['bb_pctb']
df['atr'] = atr(df['high'], df['low'], df['close'], 14)
df['vol_10'] = df['returns'].rolling(10).std()
df['mom_5'] = df['close'].pct_change(5)
df['mom_10'] = df['close'].pct_change(10)

df.dropna(inplace=True)

FEATURE_COLS = ['returns', 'rsi', 'macd_hist', 'bb_pctb', 'atr', 'vol_10', 'mom_5', 'mom_10']
WINDOW = 10  # 观察最近10天
STATE_DIM = len(FEATURE_COLS) * WINDOW
ACTION_DIM = 3  # 0=hold, 1=buy, 2=sell

# 标准化
feat_mean = df[FEATURE_COLS].mean()
feat_std = df[FEATURE_COLS].std().replace(0, 1)
df[FEATURE_COLS] = (df[FEATURE_COLS] - feat_mean) / feat_std

# 训练/测试分割
split = int(len(df) * 0.7)
train_df = df.iloc[:split].reset_index(drop=True)
test_df = df.iloc[split:].reset_index(drop=True)
print(f'训练集: {len(train_df)}, 测试集: {len(test_df)}')

In [ ]:
# ============ 交易环境 (Gymnasium风格) ============
class TradingEnv:
    """简易股票交易环境"""
    def __init__(self, data, feature_cols, window=10):
        self.data = data.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.window = window
        self.n_features = len(feature_cols)
        self.state_dim = self.n_features * self.window
        self.action_space_n = 3  # hold=0, buy=1, sell=2
        self.reset()

    def reset(self):
        self.current_step = self.window
        self.position = 0  # 0=空仓, 1=持仓
        self.total_reward = 0
        return self._get_state()

    def _get_state(self):
        start = self.current_step - self.window
        end = self.current_step
        features = self.data[self.feature_cols].iloc[start:end].values.flatten()
        return features.astype(np.float32)

    def step(self, action):
        # action: 0=hold, 1=buy, 2=sell
        current_return = self.data['returns'].iloc[self.current_step]
        reward = 0.0

        if action == 1 and self.position == 0:  # 买入
            self.position = 1
            reward = -COMMISSION_RATE
        elif action == 2 and self.position == 1:  # 卖出
            self.position = 0
            reward = -COMMISSION_RATE - STAMP_TAX_RATE

        # 持仓收益
        if self.position == 1:
            reward += current_return

        self.total_reward += reward
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        next_state = self._get_state() if not done else np.zeros(self.state_dim, dtype=np.float32)
        return next_state, reward, done, {}

In [ ]:
# ============ DQN 智能体 ============
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, x):
        return self.net(x)

class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, eps_start=1.0, eps_end=0.05):
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = eps_start
        self.eps_end = eps_end
        self.q_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.memory = deque(maxlen=5000)

    def act(self, state, explore=True):
        if explore and random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).unsqueeze(0).to(device))
            return q.argmax(1).item()

    def store(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def train_step(self, batch_size=64):
        if len(self.memory) < batch_size:
            return
        batch = random.sample(self.memory, batch_size)
        s, a, r, s2, d = zip(*batch)
        s = torch.FloatTensor(np.array(s)).to(device)
        a = torch.LongTensor(a).to(device)
        r = torch.FloatTensor(r).to(device)
        s2 = torch.FloatTensor(np.array(s2)).to(device)
        d = torch.FloatTensor(d).to(device)

        q_values = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()
        with torch.no_grad():
            q_next = self.target_net(s2).max(1)[0]
            target = r + self.gamma * q_next * (1 - d)

        loss = F.mse_loss(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def decay_epsilon(self, episode, total_episodes):
        self.epsilon = max(self.eps_end, 1.0 - episode / total_episodes)

In [ ]:
# ============ A2C 智能体 ============
class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU())
        self.actor = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, action_dim))
        self.critic = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        h = self.shared(x)
        return F.softmax(self.actor(h), dim=-1), self.critic(h)

class A2CAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.action_dim = action_dim
        self.net = ActorCriticNet(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.net.parameters(), lr=lr)
        self.log_probs = []
        self.values = []
        self.rewards = []

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs, value = self.net(s)
        if explore:
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            self.log_probs.append(dist.log_prob(action))
        else:
            action = probs.argmax(1)
        self.values.append(value.squeeze())
        return action.item()

    def store_reward(self, r):
        self.rewards.append(r)

    def train_step(self):
        if len(self.rewards) == 0:
            return
        R = 0
        returns = []
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        returns = torch.FloatTensor(returns).to(device)
        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        policy_loss = 0
        value_loss = 0
        for lp, v, R in zip(self.log_probs, self.values, returns):
            advantage = R - v.detach()
            policy_loss -= lp * advantage
            value_loss += F.mse_loss(v, R)

        loss = policy_loss + 0.5 * value_loss
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.log_probs, self.values, self.rewards = [], [], []

In [ ]:
# ============ PPO 智能体 ============
class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, eps_clip=0.2, k_epochs=4):
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.k_epochs = k_epochs
        self.action_dim = action_dim

        self.net = ActorCriticNet(state_dim, action_dim).to(device)
        self.net_old = ActorCriticNet(state_dim, action_dim).to(device)
        self.net_old.load_state_dict(self.net.state_dict())
        self.optimizer = optim.Adam(self.net.parameters(), lr=lr)

        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            probs, _ = self.net_old(s)
        if explore:
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            self.log_probs.append(dist.log_prob(action))
        else:
            action = probs.argmax(1)
        self.states.append(state)
        self.actions.append(action.item())
        return action.item()

    def store_reward(self, r):
        self.rewards.append(r)

    def train_step(self):
        if len(self.rewards) == 0:
            return
        R = 0
        returns = []
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        returns = torch.FloatTensor(returns).to(device)
        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        old_states = torch.FloatTensor(np.array(self.states)).to(device)
        old_actions = torch.LongTensor(self.actions).to(device)
        old_logprobs = torch.stack(self.log_probs).detach().to(device)

        for _ in range(self.k_epochs):
            probs, values = self.net(old_states)
            dist = torch.distributions.Categorical(probs)
            new_logprobs = dist.log_prob(old_actions)
            entropy = dist.entropy()

            ratios = torch.exp(new_logprobs - old_logprobs)
            advantages = returns - values.squeeze().detach()

            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages

            loss = -torch.min(surr1, surr2).mean() + 0.5 * F.mse_loss(values.squeeze(), returns) - 0.01 * entropy.mean()
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        self.net_old.load_state_dict(self.net.state_dict())
        self.states, self.actions, self.log_probs, self.rewards = [], [], [], []

In [ ]:
# ============ 训练三个智能体 ============
N_EPISODES = 80

dqn_agent = DQNAgent(STATE_DIM, ACTION_DIM)
a2c_agent = A2CAgent(STATE_DIM, ACTION_DIM)
ppo_agent = PPOAgent(STATE_DIM, ACTION_DIM)

reward_history = {'DQN': [], 'A2C': [], 'PPO': []}

for ep in range(N_EPISODES):
    # --- DQN ---
    env = TradingEnv(train_df, FEATURE_COLS, WINDOW)
    state = env.reset()
    ep_reward_dqn = 0
    while True:
        action = dqn_agent.act(state)
        next_state, reward, done, _ = env.step(action)
        dqn_agent.store(state, action, reward, next_state, float(done))
        dqn_agent.train_step()
        ep_reward_dqn += reward
        state = next_state
        if done:
            break
    dqn_agent.decay_epsilon(ep, N_EPISODES)
    if ep % 10 == 0:
        dqn_agent.update_target()
    reward_history['DQN'].append(ep_reward_dqn)

    # --- A2C ---
    env = TradingEnv(train_df, FEATURE_COLS, WINDOW)
    state = env.reset()
    ep_reward_a2c = 0
    while True:
        action = a2c_agent.act(state)
        next_state, reward, done, _ = env.step(action)
        a2c_agent.store_reward(reward)
        ep_reward_a2c += reward
        state = next_state
        if done:
            break
    a2c_agent.train_step()
    reward_history['A2C'].append(ep_reward_a2c)

    # --- PPO ---
    env = TradingEnv(train_df, FEATURE_COLS, WINDOW)
    state = env.reset()
    ep_reward_ppo = 0
    while True:
        action = ppo_agent.act(state)
        next_state, reward, done, _ = env.step(action)
        ppo_agent.store_reward(reward)
        ep_reward_ppo += reward
        state = next_state
        if done:
            break
    ppo_agent.train_step()
    reward_history['PPO'].append(ep_reward_ppo)

    if (ep + 1) % 20 == 0:
        print(f'Episode {ep+1}/{N_EPISODES} | '
              f'DQN: {ep_reward_dqn:.4f} | A2C: {ep_reward_a2c:.4f} | PPO: {ep_reward_ppo:.4f}')

In [ ]:
# ============ 评估与集成策略 ============
def get_agent_actions(agent, data, feature_cols, window, agent_type='dqn'):
    """获取智能体在测试集上的动作序列"""
    env = TradingEnv(data, feature_cols, window)
    state = env.reset()
    actions = [0] * window  # 前window步无动作
    while True:
        action = agent.act(state, explore=False)
        next_state, _, done, _ = env.step(action)
        actions.append(action)
        state = next_state
        if done:
            break
    return actions[:len(data)]

actions_dqn = get_agent_actions(dqn_agent, test_df, FEATURE_COLS, WINDOW)
actions_a2c = get_agent_actions(a2c_agent, test_df, FEATURE_COLS, WINDOW)
actions_ppo = get_agent_actions(ppo_agent, test_df, FEATURE_COLS, WINDOW)

# 多数投票集成
def majority_vote(a1, a2, a3):
    """多数投票: 3个智能体投票决定动作"""
    from collections import Counter
    ensemble = []
    for i in range(len(a1)):
        votes = Counter([a1[i], a2[i], a3[i]])
        ensemble.append(votes.most_common(1)[0][0])
    return ensemble

actions_ensemble = majority_vote(actions_dqn, actions_a2c, actions_ppo)

# 将动作转为回测信号: 0=hold->0, 1=buy->1, 2=sell->-1
def actions_to_signals(actions, dates):
    sig_map = {0: 0, 1: 1, 2: -1}
    signals = [sig_map.get(a, 0) for a in actions]
    # 转换为持仓状态
    position = []
    pos = 0
    for s in signals:
        if s == 1:
            pos = 1
        elif s == -1:
            pos = 0
        position.append(pos)
    return pd.Series(position, index=dates[:len(position)])

test_dates = df.index[split:split + len(test_df)]
test_prices = df['close'].iloc[split:split + len(test_df)]
test_prices.index = test_dates

sig_dqn = actions_to_signals(actions_dqn, test_dates)
sig_a2c = actions_to_signals(actions_a2c, test_dates)
sig_ppo = actions_to_signals(actions_ppo, test_dates)
sig_ensemble = actions_to_signals(actions_ensemble, test_dates)

# 回测
bt = Backtester(t_plus_1=True)
results = {}
for name, sig in [('DQN', sig_dqn), ('A2C', sig_a2c), ('PPO', sig_ppo), ('集成策略', sig_ensemble)]:
    res = bt.run(test_prices, sig)
    results[name] = res
    print(f"\n{name}:")
    for k, v in res['metrics'].items():
        print(f"  {k}: {v}")

In [ ]:
# ============ 可视化 ============
import matplotlib.pyplot as plt
from shared.visualization import set_chinese_font, plot_cumulative_comparison
set_chinese_font()

# 1. 训练奖励曲线
fig, ax = plt.subplots(figsize=(12, 5))
for name, color in [('DQN', '#F44336'), ('A2C', '#4CAF50'), ('PPO', '#2196F3')]:
    ax.plot(reward_history[name], label=name, alpha=0.8, color=color)
ax.set_title('训练奖励曲线', fontsize=14)
ax.set_xlabel('Episode')
ax.set_ylabel('累计奖励')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 策略对比
strategy_returns = {name: res['returns'] for name, res in results.items()}
plot_cumulative_comparison(strategy_returns, title='集成RL策略对比')

# 3. 集成策略详细报告
ensemble_result = results['集成策略']
plot_equity_curve(ensemble_result['equity_curve'], title='集成策略收益曲线')
plot_drawdown(ensemble_result['equity_curve'], title='集成策略回撤')
plot_metrics_table(ensemble_result['metrics'], title='集成策略绩效指标')

## 实验结果与分析

### 关键发现
1. **集成效果**: 多数投票集成策略通常比单个RL智能体更稳定，降低了单一模型过拟合的风险
2. **PPO优势**: PPO通过限制策略更新幅度，训练过程更稳定，适合金融数据的高噪声环境
3. **DQN特点**: DQN在离散动作空间表现良好，但需要经验回放缓冲区来打破样本相关性
4. **A2C特点**: A2C训练速度较快，但方差较大，收敛稳定性不如PPO

### 参数敏感性
- 训练轮数: 80轮足以观察收敛趋势，实际应用可增加至200-500轮
- 学习率: PPO使用3e-4，DQN和A2C使用1e-3
- 窗口大小: 10天的观察窗口平衡了信息量和计算效率

### 局限性
- 简化的交易环境未考虑流动性、涨跌停等约束
- 状态空间仅包含技术指标，未纳入基本面信息
- 训练轮数有限，实际应用需更充分训练